# Adaptive Lasso Prediction of Emphysema Progression
**Project:** COPDGene — Proteomics Aim 2 Prediction | **Date:** 2026-05-12

This notebook builds two adaptive lasso models that predict emphysema progression
(Phase 2 → Phase 3 CT lung density change, HU) from SomaScan blood proteomics:

| Model | Predictors |
|-------|-----------|
| **Model 1** | Proteins + baseline emphysema + clinical covariates |
| **Model 2** | Proteins + baseline emphysema + clinical covariates + **MAL** |

MAL = **Mechanically Affected Lung** — a CT-derived continuous score measuring the
proportion of lung volume under destructive mechanical stress.
Goal: quantify whether adding MAL improves proteomic prediction of emphysema progression.

## Section 0 — Background, Methods, and Notation

### Clinical Background

**Emphysema** is the progressive destruction of alveoli (tiny air sacs), measured as
CT lung density in Hounsfield Units (HU):
- Healthy lung: roughly −850 to −750 HU
- Emphysematous lung: roughly −950 to −1000 HU (more air → less dense)

**Outcome:** `Change_lung_density_vnb_P2_P3` = CT density Phase 3 − Phase 2
- Negative → emphysema **progressed**; Positive → stable/improved
- Expected range: −50 to +30 HU

**MAL (Mechanically Affected Lung):** regions where mechanical pressure gradients
deform and damage lung tissue.  Quantified as `meanmal` ∈ [0.01, 0.40].

---

### Proteomics Background

SomaScan platform measures ~5,000 blood plasma proteins as RFU values (~100–100,000;
right-skewed, log-normal).

**Log₂ transformation:** X_log₂ = log₂(RFU) normalises the distribution and makes
coefficients interpretable:
> β = expected change in outcome per **doubling** of protein level

---

### Adaptive Lasso Theory (Zou 2006)

High-dimensional problem: n ≈ 1,500 subjects, p ≈ 4,980 proteins (**p >> n**).
Classical OLS fails (singular matrix). Adaptive lasso solves:

```
argmin_β  (1/2n)||y − Xβ||²  +  λ·Σⱼ wⱼ|βⱼ|
          ──────────────────     ───────────────
            data fit (MSE)        weighted L1 penalty

  y  ∈ ℝⁿ    outcome vector (emphysema change, HU)
  X  ∈ ℝⁿˣᵖ  design matrix (proteins + covariates)
  β  ∈ ℝᵖ    coefficient vector
  λ  ∈ (0,∞) global penalty strength (chosen by CV)
  wⱼ = 1/|β̂ⱼ_ridge|  adaptive weight per predictor
```

**Oracle property:** with correct λ, adaptive lasso selects exactly the true predictors.

**Two-step procedure:**
1. **Ridge** (α=0): fit β̂_ridge on proteins → compute weights wⱼ = 1/|β̂ⱼ|
2. **Adaptive lasso** (α=1): fit with per-predictor weights → sparse β̂

---

### Evaluation Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| MSE | (1/n)Σ(yᵢ−ŷᵢ)² | Avg squared error (HU²); lower = better |
| AUC | P(ŷ_prog > ŷ_non) | 0.5=random, 0.7=acceptable, 0.8=good |
| AIC | 2k − 2ln(L̂) | Fit vs complexity; lower = better |

---

### Data Split (60/20/20)

| Set | % | n (approx) | Role |
|-----|---|------------|------|
| Training | 60 | ~900 | Fit ridge + adaptive lasso |
| Validation | 20 | ~300 | Select λ; model comparison |
| **Test** | **20** | **~300** | **Final unbiased evaluation** |

`set.seed(2024)` — same split on every run.

In [ ]:
# =============================================================================
# SECTION 1: LOAD PACKAGES
# =============================================================================

library(glmnet)  # coordinate-descent penalized regression (ridge + lasso)
library(pROC)    # ROC/AUC estimation via DeLong's non-parametric method
library(dplyr)   # tidy data manipulation
library(tibble)  # modern data frames
library(readr)   # fast CSV I/O

In [ ]:
# =============================================================================
# SECTION 2: FILE PATHS
# =============================================================================

BASE       <- here::here()                                  # project root: COPD/Journal/
PROT_PATH  <- file.path(BASE, "data/csv_exports/prot.csv")        # SomaScan RFU values
PHENO_PATH <- file.path(BASE, "data/csv_exports/pheno_MAL.csv")   # phenotypes + covariates
MAL_PATH   <- file.path(BASE, "data/Data/MAL.csv")                # MAL scores (meanmal)
META_PATH  <- file.path(BASE, "data/csv_exports/metadat5k.csv")   # protein annotations
OUT_DIR    <- file.path(BASE, "Aim2_prediction")                   # output directory

dir.create(OUT_DIR, showWarnings = FALSE, recursive = TRUE)

## Section 3 — Helper Functions

Six reusable functions encapsulate every step of the pipeline.
Both models (with/without MAL) call the same functions; `include_mal = TRUE/FALSE`
is the only difference.

In [ ]:
# -------------------------------------------------------------------
# load_and_merge_data()
# Loads proteins, phenotypes, and MAL; merges on sid; log2-transforms
# proteins; collapses rare scanner levels; defines binary outcome.
# Returns: list($df, $protein_cols)
# -------------------------------------------------------------------
load_and_merge_data <- function(prot_path, pheno_path, mal_path, meta_path) {

  prot  <- read_csv(prot_path,  show_col_types = FALSE)
  pheno <- read_csv(pheno_path, show_col_types = FALSE)
  mal   <- read_csv(mal_path,   show_col_types = FALSE) |> select(sid, meanmal)

  # inner_join: keep subjects present in ALL three sources
  df <- prot |> inner_join(pheno, by = "sid") |> inner_join(mal, by = "sid")
  message("Rows after 3-way merge: ", nrow(df))

  # SomaScan protein columns: names starting with "X" followed by a digit
  protein_cols <- grep("^X[0-9]", names(df), value = TRUE)
  message("Proteins: ", length(protein_cols))  # expect ~4,979

  # Log2-transform: X_log2 = log2(RFU)
  # WHY: RFU values are log-normal; log2 normalises and gives β interpretation
  #      as change per doubling of protein level.
  # pmax(..., 1e-6): guard against log2(0) if any RFU = 0.
  df <- df |> mutate(across(all_of(protein_cols), ~ log2(pmax(., 1e-6))))
  protein_cols_log2          <- paste0("log2_", protein_cols)
  names(df)[names(df) %in% protein_cols] <- protein_cols_log2

  # Collapse rare scanner models to "Other"
  # WHY: rare levels absent from val/test split → predict() fails on unseen factor levels
  scanner_counts <- table(df$scanner_model_clean_P2)
  rare_scanners  <- names(scanner_counts[scanner_counts < 15])
  df <- df |> mutate(scanner_model_clean_P2 =
    if_else(scanner_model_clean_P2 %in% rare_scanners, "Other", scanner_model_clean_P2))

  # Binary outcome for AUC: 1 = emphysema progressed (density decreased)
  df <- df |> mutate(progression_binary = as.integer(Change_lung_density_vnb_P2_P3 < 0))

  # Complete-case filter on outcome + all covariates (assumes data MAR)
  required_cols <- c(
    "Change_lung_density_vnb_P2_P3", "lung_density_vnb_P2", "meanmal",
    "Age_P2", "gender", "race", "ATS_PackYears_P2", "SmokCigNow_P2",
    "scanner_model_clean_P2", paste0("PC", 1:5)
  )
  df <- df |> filter(if_all(all_of(required_cols), ~ !is.na(.)))
  message("Rows after complete-case filter: ", nrow(df))  # expect ~1,500–2,000

  list(df = df, protein_cols = protein_cols_log2)
}

In [ ]:
# -------------------------------------------------------------------
# split_data()
# Reproducible 60/20/20 random split.
# Returns: list($train, $val, $test)
# -------------------------------------------------------------------
split_data <- function(df, seed = 2024, train_frac = 0.60, val_frac = 0.20) {

  set.seed(seed)   # same split on every run
  n       <- nrow(df)
  idx     <- sample(n)
  n_train <- round(train_frac * n)
  n_val   <- round(val_frac   * n)

  list(
    train = df[idx[seq_len(n_train)], ],
    val   = df[idx[seq(n_train + 1, n_train + n_val)], ],
    test  = df[idx[seq(n_train + n_val + 1, n)], ]
  )
}

In [ ]:
# -------------------------------------------------------------------
# build_design()
# Constructs the glmnet design matrix and penalty factor vector.
#
# Key design choices:
#   penalty.factor = 0   for covariates → always kept (never penalised)
#   penalty.factor = wⱼ  for proteins   → adaptive L1 penalty
#   ref_colnames         → align val/test matrices to training columns
#                          (rare factor levels get zero-filled dummy columns)
#
# Returns: list($X, $y, $y_bin, $pf, $n_fixed, $colnames)
# -------------------------------------------------------------------
build_design <- function(df, protein_cols, include_mal = FALSE,
                         adaptive_weights = NULL, ref_colnames = NULL) {

  # Fixed (unpenalised) covariates — known confounders; always kept
  fixed_vars <- c(
    "lung_density_vnb_P2",    # baseline emphysema: isolates progression signal
    "Age_P2", "gender", "race",
    "ATS_PackYears_P2",       # cumulative pack-years: key COPD driver
    "SmokCigNow_P2",          # current smoking status
    "scanner_model_clean_P2", # CT scanner model: affects HU calibration
    paste0("PC", 1:5)         # genetic ancestry PCs 1–5
  )
  if (include_mal) fixed_vars <- c(fixed_vars, "meanmal")  # Model 2 only

  # model.matrix: expands factors to dummies; "-1" drops the intercept
  # (glmnet fits its own intercept internally)
  fixed_formula <- as.formula(paste("~", paste(fixed_vars, collapse = " + "), "- 1"))
  X_fixed <- model.matrix(fixed_formula, data = df)
  n_fixed  <- ncol(X_fixed)

  X_prot <- as.matrix(df[, protein_cols, drop = FALSE])
  X      <- cbind(X_fixed, X_prot)

  # Column alignment across splits:
  # model.matrix() only includes dummy columns for levels present in THAT split.
  # If a rare scanner model is absent from val/test, its column is missing and
  # predict() crashes. Fix: zero-fill missing columns, then reorder to match training.
  if (!is.null(ref_colnames)) {
    missing_cols <- setdiff(ref_colnames, colnames(X))
    if (length(missing_cols) > 0) {
      zero_mat <- matrix(0, nrow = nrow(X), ncol = length(missing_cols),
                         dimnames = list(NULL, missing_cols))
      X <- cbind(X, zero_mat)
    }
    X <- X[, ref_colnames, drop = FALSE]
  }

  # Penalty factor: 0 for covariates (always kept), wⱼ for proteins
  pf_prot <- if (is.null(adaptive_weights)) rep(1, length(protein_cols)) else adaptive_weights
  pf      <- c(rep(0, n_fixed), pf_prot)

  list(X = X, y = df$Change_lung_density_vnb_P2_P3, y_bin = df$progression_binary,
       pf = pf, n_fixed = n_fixed, colnames = colnames(X))
}

In [ ]:
# -------------------------------------------------------------------
# compute_adaptive_weights()
# Step 1 of adaptive lasso: ridge regression on residualised outcome.
#
# WHY residualise y?  We want weights that reflect protein-specific signal.
# Fitting ridge directly on all predictors conflates covariate and protein
# effects; regressing out covariates first isolates protein signal.
#
# Returns: numeric vector wⱼ = 1/|β̂_ridge_j|, length = n_proteins
# -------------------------------------------------------------------
compute_adaptive_weights <- function(X_prot_train, y_resid_train) {

  # Ridge (alpha=0): L2 penalty Σβⱼ² — stable when p>>n; no variable selection
  # 10-fold CV within training set selects lambda.min (minimises CV-MSE)
  ridge_cv <- cv.glmnet(x = X_prot_train, y = y_resid_train,
                        alpha = 0, nfolds = 10, standardize = TRUE)

  # Extract ridge coefficients at lambda.min; drop intercept (row 1)
  ridge_coef <- as.vector(coef(ridge_cv, s = "lambda.min"))[-1]

  # Adaptive weights: wⱼ = 1/|β̂ⱼ|
  # Large |β̂| → small weight → lasso is lenient → protein survives
  # Small |β̂| → large weight → lasso is harsh  → protein zeroed out
  # pmax floor prevents Inf weights when any β̂ ≈ 0
  1 / pmax(abs(ridge_coef), 1e-10)
}

In [ ]:
# -------------------------------------------------------------------
# fit_adaptive_lasso()
# Step 2: L1-penalised regression with per-predictor adaptive weights.
# 10-fold CV within the training set selects lambda.min and lambda.1se.
#
# L1 geometry: the unit L1 ball has corners at axes → solutions are SPARSE
# (many βⱼ = exactly 0), unlike L2 (ridge) whose ball is smooth.
# Returns: cv.glmnet object
# -------------------------------------------------------------------
fit_adaptive_lasso <- function(X, y, penalty_factor, nfolds = 10) {
  cv.glmnet(x = X, y = y, alpha = 1,
            penalty.factor = penalty_factor,
            nfolds = nfolds, standardize = TRUE)
}

In [ ]:
# -------------------------------------------------------------------
# evaluate_on_split()
# Computes out-of-sample MSE and AUC at a given lambda.
# Used on validation set (model selection) and test set (final report).
# Returns: list($mse, $auc)
# -------------------------------------------------------------------
evaluate_on_split <- function(model, X_new, y_new, y_bin, lambda) {
  y_pred  <- as.vector(predict(model, newx = X_new, s = lambda))
  mse     <- mean((y_new - y_pred)^2)                              # HU²
  roc_obj <- pROC::roc(y_bin, y_pred, direction = "<", quiet = TRUE)
  auc_val <- as.numeric(pROC::auc(roc_obj))
  list(mse = mse, auc = auc_val)
}

In [ ]:
# -------------------------------------------------------------------
# refit_ols()
# Post-selection OLS refit on lasso-selected predictors.
#
# WHY refit OLS?
#   Lasso β̂ are biased toward zero by construction.
#   After lasso identifies WHICH predictors matter, OLS refits to give
#   unbiased β̂, valid standard errors, 95% CIs, and p-values.
#
# Caveat: post-selection p-values have inflated Type I error (selected
#   predictors are correlated with the outcome by construction).
#   Interpret as descriptive, not confirmatory.
#
# Returns: tibble with predictor, beta, se, ci_lower, ci_upper, p_value, AIC
# -------------------------------------------------------------------
refit_ols <- function(X_trainval, y_trainval, lasso_model, lambda,
                      protein_cols, n_fixed) {

  # Extract lasso coefficients at chosen lambda; drop intercept
  lasso_coef <- as.vector(coef(lasso_model, s = lambda))[-1]

  # Identify selected proteins (non-zero lasso coef among protein columns)
  protein_indices   <- seq(n_fixed + 1, length(lasso_coef))
  selected_proteins <- protein_indices[lasso_coef[protein_indices] != 0]
  selected_all      <- c(seq_len(n_fixed), selected_proteins)

  message("Proteins selected: ", length(selected_proteins))

  X_sel    <- X_trainval[, selected_all, drop = FALSE]
  ols_fit  <- lm(y_trainval ~ X_sel)   # OLS on selected predictors

  ols_summary <- summary(ols_fit)$coefficients   # Estimate, SE, t, p
  ols_ci      <- confint(ols_fit, level = 0.95)  # 95% CI via t-distribution
  ols_aic     <- AIC(ols_fit)                    # 2k − 2ln(L̂)

  # Use rownames(ols_summary) as labels: lm() silently drops rank-deficient
  # columns, so nrow(summary) may be < ncol(X_sel)+1. rownames keeps everything
  # in sync. Strip the "X_sel" prefix lm() auto-prepends to matrix column names.
  clean_names <- sub("^X_sel", "", rownames(ols_summary))

  tibble(
    predictor = clean_names,
    beta      = ols_summary[, "Estimate"],
    se        = ols_summary[, "Std. Error"],
    ci_lower  = ols_ci[rownames(ols_summary), 1],
    ci_upper  = ols_ci[rownames(ols_summary), 2],
    p_value   = ols_summary[, "Pr(>|t|)"],
    AIC       = ols_aic
  )
}

In [ ]:
# -------------------------------------------------------------------
# compute_prs()
# Computes the Proteomic Risk Score for all subjects in the full cohort.
#
# PRS_i = Σⱼ β̂_OLS_j × log₂(protein_ij)    for selected proteins j
#
# Standardised: PRS_scaled = (PRS − mean) / sd
# Interpretation: 1 unit = 1 SD increase in proteomic risk
# Returns: numeric vector (length = nrow(df))
# -------------------------------------------------------------------
compute_prs <- function(df, ols_coefs, protein_cols) {

  protein_rows <- ols_coefs |> filter(predictor %in% protein_cols)
  if (nrow(protein_rows) == 0) stop("No protein coefficients found in OLS output.")

  selected_proteins <- protein_rows$predictor
  protein_betas     <- setNames(protein_rows$beta, selected_proteins)

  X_prot_sel <- as.matrix(df[, selected_proteins, drop = FALSE])
  prs_raw    <- as.vector(X_prot_sel %*% protein_betas)  # weighted sum

  # Standardise so β_PRS = HU change per 1-SD increase in proteomic risk
  (prs_raw - mean(prs_raw, na.rm = TRUE)) / sd(prs_raw, na.rm = TRUE)
}

## Section 4 — Analysis Pipeline

In [ ]:
# --- 4.1: Load and preprocess all data ---
message("=== Loading and preprocessing data ===")
data_obj     <- load_and_merge_data(PROT_PATH, PHENO_PATH, MAL_PATH, META_PATH)
df_full      <- data_obj$df
protein_cols <- data_obj$protein_cols

In [ ]:
# --- 4.2: Split into train / validation / test (60/20/20) ---
message("=== Splitting data 60/20/20 ===")
splits <- split_data(df_full, seed = 2024)

message("Train n = ", nrow(splits$train),
        "  |  Val n = ", nrow(splits$val),
        "  |  Test n = ", nrow(splits$test))

In [ ]:
# --- 4.3: Initial design matrix (uniform weights) for ridge pre-step ---
message("=== Computing adaptive weights via ridge regression ===")
design_m1_train_init <- build_design(splits$train, protein_cols,
                                     include_mal = FALSE, adaptive_weights = NULL)

# Residualise y w.r.t. covariates to isolate protein-specific ridge signal
X_cov_train  <- design_m1_train_init$X[, seq_len(design_m1_train_init$n_fixed), drop = FALSE]
X_prot_train <- design_m1_train_init$X[, -seq_len(design_m1_train_init$n_fixed), drop = FALSE]
y_train      <- design_m1_train_init$y
y_resid      <- residuals(lm.fit(X_cov_train, y_train))

# Ridge → adaptive weights
adaptive_weights <- compute_adaptive_weights(X_prot_train, y_resid)
message("Adaptive weights: min = ", round(min(adaptive_weights), 4),
        "  max = ", round(max(adaptive_weights), 4))
message("Infinite weights: ", sum(is.infinite(adaptive_weights)))  # must be 0

# --- 4.4–4.5: Rebuild all design matrices with adaptive weights ---
message("=== Rebuilding design matrices with adaptive weights ===")

design_m1_train <- build_design(splits$train, protein_cols,
                                include_mal = FALSE, adaptive_weights = adaptive_weights)
design_m2_train <- build_design(splits$train, protein_cols,
                                include_mal = TRUE,  adaptive_weights = adaptive_weights)

design_m1_val  <- build_design(splits$val,  protein_cols, include_mal = FALSE,
                               adaptive_weights = adaptive_weights,
                               ref_colnames = design_m1_train$colnames)
design_m2_val  <- build_design(splits$val,  protein_cols, include_mal = TRUE,
                               adaptive_weights = adaptive_weights,
                               ref_colnames = design_m2_train$colnames)

design_m1_test <- build_design(splits$test, protein_cols, include_mal = FALSE,
                               adaptive_weights = adaptive_weights,
                               ref_colnames = design_m1_train$colnames)
design_m2_test <- build_design(splits$test, protein_cols, include_mal = TRUE,
                               adaptive_weights = adaptive_weights,
                               ref_colnames = design_m2_train$colnames)

In [ ]:
# --- 4.6: Fit adaptive lasso on training set ---
message("=== Fitting adaptive lasso ===")
lasso_m1 <- fit_adaptive_lasso(design_m1_train$X, design_m1_train$y, design_m1_train$pf)
lasso_m2 <- fit_adaptive_lasso(design_m2_train$X, design_m2_train$y, design_m2_train$pf)

In [ ]:
# --- 4.7: Validation — select lambda, compare models ---
message("=== Validation: lambda selection ===")

val_m1_min <- evaluate_on_split(lasso_m1, design_m1_val$X,
                                design_m1_val$y, design_m1_val$y_bin, "lambda.min")
val_m1_1se <- evaluate_on_split(lasso_m1, design_m1_val$X,
                                design_m1_val$y, design_m1_val$y_bin, "lambda.1se")
val_m2_min <- evaluate_on_split(lasso_m2, design_m2_val$X,
                                design_m2_val$y, design_m2_val$y_bin, "lambda.min")
val_m2_1se <- evaluate_on_split(lasso_m2, design_m2_val$X,
                                design_m2_val$y, design_m2_val$y_bin, "lambda.1se")

cat(sprintf("Model 1  lambda.min  MSE=%.3f  AUC=%.3f\n", val_m1_min$mse, val_m1_min$auc))
cat(sprintf("Model 1  lambda.1se  MSE=%.3f  AUC=%.3f\n", val_m1_1se$mse, val_m1_1se$auc))
cat(sprintf("Model 2  lambda.min  MSE=%.3f  AUC=%.3f\n", val_m2_min$mse, val_m2_min$auc))
cat(sprintf("Model 2  lambda.1se  MSE=%.3f  AUC=%.3f\n", val_m2_1se$mse, val_m2_1se$auc))

# Select lambda: prefer lambda.min if it achieves equal or better validation AUC
lambda_m1 <- if (val_m1_min$auc >= val_m1_1se$auc) "lambda.min" else "lambda.1se"
lambda_m2 <- if (val_m2_min$auc >= val_m2_1se$auc) "lambda.min" else "lambda.1se"
message("Selected lambda — Model 1: ", lambda_m1, "  |  Model 2: ", lambda_m2)

In [ ]:
# --- 4.8: OLS refit on train + validation combined ---
# Using 80% of data maximises power for stable β̂, CI, and AIC comparison
message("=== OLS refit on train + validation ===")

df_trainval  <- bind_rows(splits$train, splits$val)
design_m1_tv <- build_design(df_trainval, protein_cols, include_mal = FALSE,
                             adaptive_weights = adaptive_weights,
                             ref_colnames = design_m1_train$colnames)
design_m2_tv <- build_design(df_trainval, protein_cols, include_mal = TRUE,
                             adaptive_weights = adaptive_weights,
                             ref_colnames = design_m2_train$colnames)

ols_m1 <- refit_ols(design_m1_tv$X, design_m1_tv$y, lasso_m1, lambda_m1,
                    protein_cols, design_m1_tv$n_fixed)
ols_m2 <- refit_ols(design_m2_tv$X, design_m2_tv$y, lasso_m2, lambda_m2,
                    protein_cols, design_m2_tv$n_fixed)

In [ ]:
# --- 4.9: Final test set evaluation (reported metrics) ---
message("=== Test set evaluation ===")

test_m1 <- evaluate_on_split(lasso_m1, design_m1_test$X,
                             design_m1_test$y, design_m1_test$y_bin, lambda_m1)
test_m2 <- evaluate_on_split(lasso_m2, design_m2_test$X,
                             design_m2_test$y, design_m2_test$y_bin, lambda_m2)

n_prot_m1 <- ols_m1 |> filter(predictor %in% protein_cols) |> nrow()
n_prot_m2 <- ols_m2 |> filter(predictor %in% protein_cols) |> nrow()

comparison_table <- tibble(
  model               = c("Model 1 (no MAL)", "Model 2 (+MAL)"),
  n_proteins_selected = c(n_prot_m1, n_prot_m2),
  test_MSE            = c(test_m1$mse, test_m2$mse),
  test_AUC            = c(test_m1$auc, test_m2$auc),
  AIC_OLS_refit       = c(unique(ols_m1$AIC), unique(ols_m2$AIC)),
  lambda_chosen       = c(lambda_m1, lambda_m2)
)

print(comparison_table)

In [ ]:
# --- 4.10: Proteomic Risk Score ---
# Derived from Model 2 coefficients; applied to the full cohort
message("=== Computing Proteomic Risk Score (PRS) ===")

prs_scaled <- compute_prs(df_full, ols_m2, protein_cols)
message("PRS: mean=", round(mean(prs_scaled), 3),
        "  sd=",  round(sd(prs_scaled), 3),
        "  range [", round(min(prs_scaled), 2), ", ", round(max(prs_scaled), 2), "]")

# Association: PRS → emphysema progression (adjusted for known confounders)
prs_model   <- lm(Change_lung_density_vnb_P2_P3 ~ prs_scaled + lung_density_vnb_P2 +
                    Age_P2 + gender + race + ATS_PackYears_P2 + SmokCigNow_P2 +
                    scanner_model_clean_P2 + PC1 + PC2 + PC3 + PC4 + PC5,
                  data = df_full)
prs_summary <- summary(prs_model)$coefficients
prs_ci      <- confint(prs_model)

prs_assoc_table <- tibble(
  predictor = rownames(prs_summary),
  beta      = prs_summary[, "Estimate"],
  se        = prs_summary[, "Std. Error"],
  ci_lower  = prs_ci[, 1],
  ci_upper  = prs_ci[, 2],
  p_value   = prs_summary[, "Pr(>|t|)"]
)

prs_row <- filter(prs_assoc_table, predictor == "prs_scaled")
message("PRS association: beta=", round(prs_row$beta, 3),
        "  95% CI [", round(prs_row$ci_lower, 3), ", ", round(prs_row$ci_upper, 3), "]",
        "  p=", signif(prs_row$p_value, 3))

## Section 5 — Save Outputs

| File | Contents |
|------|----------|
| `Aim2_model_comparison.csv` | AUC, MSE, AIC, N proteins — Model 1 vs 2 on test set |
| `Aim2_lasso_coefs_m1.csv` | Selected proteins + β, SE, CI, p (Model 1 OLS refit) |
| `Aim2_lasso_coefs_m2.csv` | Selected proteins + β, SE, CI, p + MAL row (Model 2) |
| `Aim2_proteomic_risk_score.csv` | Per-subject PRS_scaled + outcome + split |
| `Aim2_PRS_progression_association.csv` | Full lm() table for PRS → progression |

In [ ]:
# =============================================================================
# SECTION 5: SAVE OUTPUTS
# =============================================================================

write_csv(comparison_table, file.path(OUT_DIR, "Aim2_model_comparison.csv"))
message("Saved: Aim2_model_comparison.csv")

write_csv(ols_m1, file.path(OUT_DIR, "Aim2_lasso_coefs_m1.csv"))
message("Saved: Aim2_lasso_coefs_m1.csv")

write_csv(ols_m2, file.path(OUT_DIR, "Aim2_lasso_coefs_m2.csv"))
message("Saved: Aim2_lasso_coefs_m2.csv")

prs_output <- tibble(
  sid                           = df_full$sid,
  prs_scaled                    = prs_scaled,
  Change_lung_density_vnb_P2_P3 = df_full$Change_lung_density_vnb_P2_P3,
  progression_binary            = df_full$progression_binary,
  split = case_when(
    df_full$sid %in% splits$train$sid ~ "train",
    df_full$sid %in% splits$val$sid   ~ "validation",
    df_full$sid %in% splits$test$sid  ~ "test"
  )
)
write_csv(prs_output, file.path(OUT_DIR, "Aim2_proteomic_risk_score.csv"))
message("Saved: Aim2_proteomic_risk_score.csv")

write_csv(prs_assoc_table, file.path(OUT_DIR, "Aim2_PRS_progression_association.csv"))
message("Saved: Aim2_PRS_progression_association.csv")

message("=== Analysis complete ===")